# Treinamento e seleção do adaptador

Este notebook analisa os logs reais dos treinamentos locais. O treinamento foi executado pela autora no Windows e não é refeito aqui. Cada célula mostra dados extraídos dos JSONs originais. Para nova execução de treino, use o procedimento documentado no README, com outra pasta de saída.


In [1]:
from pathlib import Path
import json
import sys

RAIZ = Path.cwd()
if not (RAIZ / "base" / "conhecimento.json").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "base" / "conhecimento.json").exists(), "Abra a partir da raiz do projeto ou da pasta notebooks."
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

def ler_json(caminho):
    return json.loads((RAIZ / caminho).read_text(encoding="utf-8-sig"))


In [2]:
for numero in [1, 2, 3]:
    pasta = f"artefatos/lora_experimento_{numero:02}"
    log = ler_json(f"{pasta}/treinamento.json")
    print(f"\nEXPERIMENTO {numero:02}")
    print("Perda inicial:", round(log["perda_validacao_inicial"], 6))
    for epoca in log["historico"]:
        print("Época", epoca["epoca"], "| treino", round(epoca["perda_treino"], 6), "| validação", round(epoca["perda_validacao"], 6))
    print("Melhor época:", log["melhor_epoca"])



EXPERIMENTO 01
Perda inicial: 0.472709
Época 1 | treino 0.240461 | validação 0.560837
Época 2 | treino 0.055074 | validação 0.031521
Época 3 | treino 0.006237 | validação 0.27122
Melhor época: 2

EXPERIMENTO 02
Perda inicial: 0.472709
Época 1 | treino 0.138327 | validação 0.006456
Época 2 | treino 0.00215 | validação 0.002304
Época 3 | treino 0.000414 | validação 0.002076
Melhor época: 3

EXPERIMENTO 03
Perda inicial: 0.224634
Época 1 | treino 0.173716 | validação 0.030333
Época 2 | treino 0.063615 | validação 0.00932
Época 3 | treino 0.02743 | validação 0.009827
Melhor época: 2


In [3]:
config = ler_json("artefatos/lora_experimento_03/adapter_config.json")
log = ler_json("artefatos/lora_experimento_03/treinamento.json")
print("Modelo base:", config["base_model_name_or_path"])
print("Rank:", config["r"], "| Alpha:", config["lora_alpha"], "| Dropout:", config["lora_dropout"])
print("Módulos:", sorted(config["target_modules"]))
print("Taxa de aprendizado:", log["taxa_aprendizado"])
print("Tempo registrado:", round(log["segundos_decorridos"], 1), "segundos")
print("Revisão do modelo base fixada:", config["revision"])


Modelo base: Qwen/Qwen3-0.6B
Rank: 8 | Alpha: 16 | Dropout: 0.05
Módulos: ['q_proj', 'v_proj']
Taxa de aprendizado: 0.0001
Tempo registrado: 453.3 segundos
Revisão do modelo base fixada: None


In [4]:
for nome in ["treino_json", "validacao_json"]:
    dados = ler_json(f"artefatos/lora_experimento_03/{nome}.json")
    respondidos = sum(item["resposta"]["status"] == "respondido" for item in dados)
    recusas = sum(item["resposta"]["status"] == "nao_consta" for item in dados)
    print(nome, "| total", len(dados), "| respondidos", respondidos, "| recusas", recusas)
    for item in dados:
        resposta = item["resposta"]
        docs = {d["fonte_id"]: d["texto"] for d in item["documentos"]}
        for evidencia in resposta["evidencias"]:
            assert evidencia["fonte_id"] in docs
            assert evidencia["citacao"] in docs[evidencia["fonte_id"]]
print("Citações dos gabaritos presentes literalmente nos contextos.")


treino_json | total 32 | respondidos 14 | recusas 18
validacao_json | total 8 | respondidos 4 | recusas 4
Citações dos gabaritos presentes literalmente nos contextos.


## Limites da seleção

A época 2 do experimento 03 teve menor perda de validação que a época 3. O conjunto de oito exemplos fictícios mede um comportamento específico de estrutura e resposta, não a generalização ampla. A avaliação posterior também usa fatos que apareceram no treino de domínio, com novas formulações de perguntas. A queda de perda não equivale a eliminação de alucinações.

O experimento 01 teve 16 exemplos. O 02 teve 28. O 03 teve 32 e treinou JSON com documentos do Reencontro. Esses experimentos diferem em dados, número de passos e formato, limitando atribuição causal a uma única mudança.
